# Basic Catia .CATPart Extraction Example

This is an example of how a user would perform the following steps:
- Import the istari-digital-client from PyPI
- Instantiate an instance of the istari-digital-client Client class
- Upload a Catia model
- Extract the Catia model
- View the extracted artifacts

## Install dependencies from PyPI

In [ ]:
!pip install istari-digital-client
!pip install python-dotenv

## Instantiate the istari-digital-client Client class

To interact with Istari Digital, we need to create an instance of the istari-digital-client Client class.

The Client class takes a Configuration object that contains the following parameters:
- registry_url (required): The URL of the Istari Digital Registry Service
- registry_auth_token (required): The authentication token to use to authenticate with the Istari Digital Registry Service
- retry_enabled (optional): Whether to retry failed requests.  Defaults to True
- retry_max_attempts (optional): The maximum number of retry attempts.  Defaults to 3
- retry_min_interval_millis (optional): The minimum interval between retry attempts in milliseconds.
- retry_max_interval_millis (optional): The maximum interval between retry attempts in milliseconds.
- retry_jitter_enabled (optional): Whether to use jitter when retrying failed requests.  Defaults to True
- filesystem_cache_enabled (optional): Whether to use the filesystem cache.  Defaults to True
- filesystem_cache_root (optional): The root directory of the filesystem cache.
- filesystem_cache_clean_on_exit (optional): Whether to clean the filesystem cache on exit.  Defaults to True
- multipart_chunksize (optional): The chunk size to use when uploading files.
- multipart_threshold (optional): The threshold size to use when uploading files.

We MUST set the registry_url and registry_auth_token parameters to interact with Istari Digital.
In this example, we are following best practices and using environment variables to store the
registry_url and registry_auth_token. The environment variables are loaded using the python-dotenv package.

The configuration parameters are then used to instantiate the Configuration class.
Once instantiated, the configuration object is passed to the istari-digital-client Client class to create
an instance of the Client class.

In [ ]:
import istari_digital_client as istari_digital
import os
from dotenv import load_dotenv

load_dotenv()

dev_auth_token = os.getenv("DEV_ACCESS_TOKEN")
assert dev_auth_token is not None

dev_registry_url = os.getenv("REGISTRY_URL")
assert dev_registry_url is not None

configuration = istari_digital.Configuration(
    registry_url=dev_registry_url,
    registry_auth_token=dev_auth_token,
)
assert configuration is not None

client = istari_digital.Client(
    config = configuration
)
assert client is not None

## Upload the Catia .CATPart file to the Istari Digital Registry Service

Before extracting the Catia .CATPart file, the file must be uploaded to the Istari Digital Registry Service.
To do this, we use the add_model method of the client object.
The add_model method takes the following parameters:
- path: The path to the file to upload.
- description: An optional description of the file
- version_name: An optional version name of the file
- external_identifier: An optional external identifier of the file
- display_name: An optional display name of the file.  Useful for displaying in the UI

The parameters are then passed to the add_model method of the client object to upload the file.
The add_model method returns a Model object that contains the metadata of the uploaded file.

We can then validate that the file was successfully uploaded by accessing the properties of
the Model object that is returned.

In [ ]:
from pathlib import Path

file_path = Path("./files/WING.CATPart")

description = "Catia model v1"
version_name = "v1"
external_identifier = "catia_model_v1"
display_name = "Catia Model v1"

catia_model = client.add_model(
    path=file_path,
    description=description,
    version_name=version_name,
    external_identifier=external_identifier,
    display_name=display_name,
)

assert catia_model is not None
assert isinstance(catia_model, istari_digital.Model)

assert catia_model.description == description
assert catia_model.version_name == version_name
assert catia_model.external_identifier == external_identifier
assert catia_model.display_name == display_name
assert catia_model.read_bytes() == file_path.read_bytes()

## Extract the uploaded Catia .CATPart file

To extract the uploaded Catia .CATPart file, the following params are needed:
- model_id: The id of the model to extract
- function: The extraction function to run
- tool_name: The tool that is needed to extract the Catia file
- tool_version: The version of the tool
- operating_system: The operating system that the tool will run on

The extract parameters are then passed to the add_job method of the client object to begin the extraction job.

Once the job is created, we poll the job until it is completed.  The job is polled by checking the JobStatusName of the job.

In [ ]:
import time

model_id = catia_model.id
function = "@istari:extract"
tool_name = "dassault_catia_v5"
tool_version = "6R2023"
operating_system = "Windows 10"

job = client.add_job(
    model_id=model_id,
    function=function,
    tool_name=tool_name,
    tool_version=tool_version,
    operating_system=operating_system,
)

assert job is not None
assert isinstance(job, istari_digital.Job)

while job.status.name not in [istari_digital.JobStatusName.COMPLETED, istari_digital.JobStatusName.FAILED]:
    time.sleep(5)
    job = client.get_job(job.id)

assert job.status.name == istari_digital.JobStatusName.COMPLETED

## Clean up the uploaded file

The uploaded file can be archived using the archive_model method of the client object.
The archive_model method takes the following parameters:
- model_id: The id of the model to archive
- archive: An Archive object that contains the reason for archiving the model

This is useful for cleaning up the files if you are going to do multiple runs of the notebook.

In [ ]:
archive_reason = istari_digital.Archive(
    reason="This file was used for an example"
)

archived_model = client.archive_model(
    model_id=model_id,
    archive=archive_reason,
)

assert archived_model is not None
assert archived_model.archive_status.name == istari_digital.ArchiveStatusName.ARCHIVED